# A Theorem Prover for First-Order Logic without Equality

This notebook implements a resolution-based theorem prover in Java.

We need the parser for first order formulas, hence we load it. Formulas are represented as trees of records.  In our parser, variables start with an uppercase letter, and predicate and function symbols start with a lowercase letter.

The resolution calculus works with clauses. The file `FOL-CNF.jsh` implements the function `normalize(f)` that turns a formula `f` into a set of clauses.

The file `Unification.jsh` implements unification via the algorithm of Martelli and Montanari.

In [ ]:
%load ../lib/RecursiveSet.jsh
%load FOL-Parser.jsh
%load FOL-CNF.jsh
%load Unification.jsh

## Literals and Clauses

A *literal* is either an atomic formula or the negation of an atomic formula, i.e. it is either an object of the record class `Atom` or it has the form `Not(Atom(...))`.  A clause is a set of literals and is therefore represented as a `RecursiveSet<Formula>`.

In the TypeScript version of this notebook, a number of auxiliary functions were needed in order to unpack the tuples representing terms and literals.  In Java, these functions are not needed, since we can use *record patterns* to access the components of terms and formulas.

Given a literal `l`, the function `isNegativeLiteral` checks whether `l` is a negated atomic formula.

In [ ]:
boolean isNegativeLiteral(Formula l) {
    return l instanceof Not;
}

Given a literal `l`, the function `atomOf` strips away any negation, returning the underlying positive atomic formula.

In [ ]:
Atom atomOf(Formula l) {
    return switch (l) {
        case Not(Atom a) -> a;
        case Atom a      -> a;
        default -> throw new IllegalArgumentException("not a literal: " + l);
    };
}

## Variable Extraction and Substitution

The function `collectVariablesTerm` computes the variables occurring in a term.

In [ ]:
RecursiveSet<String> collectVariablesTerm(Term t) {
    return switch (t) {
        case Var(var x)           -> set(x);
        case App(var _, var args) -> flatMap(args, arg -> collectVariablesTerm(arg));
        default -> throw new IllegalArgumentException("unknown term: " + t);
    };
}

The function `collectVariablesLit` computes the variables occurring in a literal.

In [ ]:
RecursiveSet<String> collectVariablesLit(Formula l) {
    return flatMap(atomOf(l).args(), arg -> collectVariablesTerm(arg));
}

Given a clause `C`, the function `collectVariablesClause(C)` computes the set of all variables occurring in `C`.

In [ ]:
RecursiveSet<String> collectVariablesClause(RecursiveSet<Formula> C) {
    return flatMap(C, l -> collectVariablesLit(l));
}

The function `renameVariables(f, g)` takes two clauses `f` and `g` and renames the variables in the clause `f` so that they are different from the variables occurring in `g`.  The renaming is done with the function `applyFormula` from the file `FOL-CNF.jsh`, which applies a substitution to a formula.  The list `asciiUppercase` of all upper case letters is also defined in this file.

In [ ]:
RecursiveSet<Formula> renameVariables(RecursiveSet<Formula> f, RecursiveSet<Formula> g) {
    var oldVars   = new ArrayList<>(collectVariablesClause(f));
    var gVars     = collectVariablesClause(g);
    var freshVars = asciiUppercase.stream().filter(v -> !gVars.contains(v)).toList();
    Map<String, Term> sigma = new LinkedHashMap<>();
    for (int i = 0; i < oldVars.size() && i < freshVars.size(); i++) {
        sigma.put(oldVars.get(i), new Var(freshVars.get(i)));
    }
    return f.map(l -> applyFormula(l, sigma));
}

The function `unify` from the file `Unification.jsh` unifies two terms.  In order to unify two atomic formulas $p(s_1,\cdots,s_n)$ and $p(t_1,\cdots,t_n)$, the function `atomToTerm` converts an atomic formula into a term with the same structure.

In [ ]:
Term atomToTerm(Atom a) {
    return new App(a.predicate(), a.args());
}

## A Calculus for First Order Logic

The resolution rule is an inference rule that is defined as follows: If
 * $C_1$ and $C_2$ are clauses from first order logic,
 * $p(s_1,\cdots,s_n)$ and $p(t_1,\cdots,t_n)$ are atomic formulas,
 * the syntactical equation $p(s_1,\cdots,s_n) \doteq p(t_1,\cdots,t_n)$ is solvable and
     $$ \mu = \mathtt{mgu}\bigl(p(s_1,\cdots,s_n), p(t_1,\cdots,t_n)\bigr), $$
then
$$\frac{C_1 \cup\{ p(s_1,\cdots,s_n)\} \quad\quad \{\neg p(t_1,\cdots,t_n)\} \cup C_2}{
                 C_1\mu \cup C_2\mu} 
$$
is an application of the resolution rule.

Given a two clauses `C1` and `C2`, the function `resolve(C1, C2)` computes a set of all clauses that can be inferred from `C1` and `C2` by applying the resolution rule.

In [ ]:
RecursiveSet<RecursiveSet<Formula>> resolve(RecursiveSet<Formula> C1, RecursiveSet<Formula> C2) {
    var C2New = renameVariables(C2, C1);
    RecursiveSet<RecursiveSet<Formula>> result = set();
    for (var L1 : C1) {
        for (var L2 : C2New) {
            if (isNegativeLiteral(L1) != isNegativeLiteral(L2)) {
                var mu = unify(atomToTerm(atomOf(L1)), atomToTerm(atomOf(L2)));
                if (mu != null) {
                    var resolvent = C1.difference(set(L1)).union(C2New.difference(set(L2)));
                    result.add(resolvent.map(l -> applyFormula(l, mu)));
                }
            }
        }
    }
    return result;
}

## Factorization

A calculus which only contains the resolution rule is not complete. We also need the factorization rule. If
1. $C$ is a clause from first order logic,
2. $p(s_1,\cdots,s_n)$ and $p(t_1,\cdots,t_n)$ are atomic formulas,
3. the syntactical equation $p(s_1,\cdots,s_n)  \doteq p(t_1,\cdots,t_n)$ is solvable and 
     $$\mu = \mathtt{mgu}\bigl(p(s_1,\cdots,s_n), p(t_1,\cdots,t_n)\bigr), $$
then both 
$\displaystyle \frac{C \cup \bigl\{p(s_1,\cdots,s_n),\, p(t_1,\cdots,t_n)\bigl\}}{C\mu \cup \bigl\{p(s_1,\cdots,s_n)\mu\bigr\} } $ 
and 
$\displaystyle \frac{C \cup \bigl\{ \neg p(s_1,\cdots,s_n),\, \neg p(t_1,\cdots,t_n)\bigl\}}{C\mu \cup \bigl\{\neg p(s_1,\cdots,s_n)\mu\bigr\} }
$
are applications of the factorization rule.

The function `factorize(C)` takes a clause `C` from first order logic and computes all clauses that can be derived from `C` via factorization.

In [ ]:
RecursiveSet<RecursiveSet<Formula>> factorize(RecursiveSet<Formula> C) {
    RecursiveSet<RecursiveSet<Formula>> result = set();
    var arrC = new ArrayList<>(C);
    for (int i = 0; i < arrC.size(); i++) {
        for (int j = i + 1; j < arrC.size(); j++) {
            var L1 = arrC.get(i);
            var L2 = arrC.get(j);
            if (isNegativeLiteral(L1) == isNegativeLiteral(L2)) {
                var mu = unify(atomToTerm(atomOf(L1)), atomToTerm(atomOf(L2)));
                if (mu != null) {
                    result.add(C.map(l -> applyFormula(l, mu)));
                }
            }
        }
    }
    return result;
}

## Automatic Theorem Proving

An object of the record class `Inference` stores a clause together with the *reason* for this clause.  The reason is the list of clauses from which the clause has been derived: If the clause results from factorization, this list contains one clause.  If it results from resolution, it contains two clauses.

Given a set of clauses `Clauses`, the function `infere(Clauses)` returns a list of all possible clauses that result from the resolution of two clauses or the factorization of a single clause within the set.

In [ ]:
record Inference(RecursiveSet<Formula> clause, List<RecursiveSet<Formula>> reason) {}

In [ ]:
List<Inference> infere(RecursiveSet<RecursiveSet<Formula>> Clauses) {
    List<Inference> result = new ArrayList<>();
    for (var C1 : Clauses) {
        for (var C2 : Clauses) {
            for (var C : resolve(C1, C2)) {
                result.add(new Inference(C, List.of(C1, C2)));
            }
        }
        for (var C : factorize(C1)) {
            result.add(new Inference(C, List.of(C1)));
        }
    }
    return result;
}

The function `saturate(Cs)` takes a set of clauses `Cs` as input and tries to infer the empty clause. If it is not possible to infer the empty clause, the function runs until memory is exhausted.  The function returns a map that maps every derived clause to the reason why it has been derived.

In [ ]:
Map<RecursiveSet<Formula>, List<RecursiveSet<Formula>>> saturate(RecursiveSet<RecursiveSet<Formula>> Cs) {
    var Clauses = Cs.clone();
    int cnt     = 1;
    Map<RecursiveSet<Formula>, List<RecursiveSet<Formula>>> Reasons = new HashMap<>();
    RecursiveSet<Formula> emptyClause = set();
    while (!Clauses.contains(emptyClause)) {
        boolean added = false;
        for (var inference : infere(Clauses)) {
            if (!Clauses.contains(inference.clause())) {
                Reasons.put(inference.clause(), inference.reason());
                Clauses.add(inference.clause());
                added = true;
            }
        }
        System.out.println("cnt = " + cnt + ", number of clauses: " + Clauses.size());
        if (!added) {
            break;
        }
        cnt += 1;
    }
    return Reasons;
}

## Proof Reconstruction

In the TypeScript version, a number of functions were needed to convert terms, literals, and clauses into strings.  In Java, the method `toString` of the records and of the class `RecursiveSet` already produces a readable representation, e.g. the clause $\{\neg \texttt{red}(X), \texttt{canFly}(X)\}$ is printed as `{¬red(X), canFly(X)}`.

The function `updateProof` merges a generated proof segment `P2` into `P1` while guaranteeing lines are not duplicated.

In [ ]:
List<String> updateProof(List<String> P1, List<String> P2) {
    List<String> result = new ArrayList<>(P1);
    for (var line : P2) {
        if (!result.contains(line)) {
            result.add(line);
        }
    }
    return result;
}

Given a map `Reasons` and a clause `clause`, the function `constructProof` traverses backwards along the reason graph and returns a text-based proof, i.e. a list of strings, leading up to `clause`.

In [ ]:
List<String> constructProof(RecursiveSet<Formula> clause,
                            Map<RecursiveSet<Formula>, List<RecursiveSet<Formula>>> Reasons) {
    var reason = Reasons.get(clause);
    if (reason == null) {
        return new ArrayList<>(List.of("Axiom:       " + clause));
    }
    if (reason.size() == 1) {
        var C     = reason.get(0);
        var Proof = constructProof(C, Reasons);
        Proof.add("Factorization: " + C + " \n⊢            " + clause);
        return Proof;
    }
    var C1    = reason.get(0);
    var C2    = reason.get(1);
    var Proof = updateProof(constructProof(C1, Reasons), constructProof(C2, Reasons));
    Proof.add("Resolution:  " + C1 + ",\n             " + C2 + "  \n⊢            " + clause);
    return Proof;
}

## Testing with the Red Dragons

According to Uwe Schöning, the theory of red dragons is given by the following axioms:

1. Every dragon is happy if all its children can fly.
2. All red dragons can fly.
3. The children of red dragons are themselves red.

We will show that these axioms imply that all red dragons are happy. To this end, the formula stating that all red dragons can fly is negated. Then we will show that the set consisting of the negated formula together with the axioms is inconsistent.

In [ ]:
var s1 = "∀X:(∀Y:(child(Y, X) → canFly(Y)) → happy(X))";
var s2 = "∀X:(red(X) → canFly(X))";
var s3 = "∀X:(red(X) → ∀Y:(child(Y, X) → red(Y)))";
var s4 = "¬∀X:(red(X) → happy(X))";

var Clauses = normalize(parse(s1)).union(normalize(parse(s2)))
                                  .union(normalize(parse(s3)))
                                  .union(normalize(parse(s4)));
for (var C : Clauses) {
    System.out.println(C);
}

In [ ]:
System.out.println("Saturating...");
var Reasons = saturate(Clauses);
RecursiveSet<Formula> emptyClause = set();
var Proof = constructProof(emptyClause, Reasons);

Proof.forEach(line -> System.out.println(line));